# Amazon Textract 문서 AI 실습 과정

## 과정 개요
Amazon Textract는 머신러닝을 활용해 스캔 문서, PDF, 이미지에서 텍스트와 데이터를 자동으로 추출하는 완전관리형 서비스입니다.
단순 OCR을 넘어 **표, 양식(Key-Value), 쿼리 기반 추출, 영수증/신분증 분석**까지 지원합니다.

---

## 📅 4시간 커리큘럼

| # | 실습 | API | 소요시간 | 핵심 개념 |
|---|---|---|---|---|
| 0 | **과정 소개 & 환경 설정** | - | 20분 | boto3, Block 구조 이해 |
| 1 | **텍스트 감지** | `detect_document_text` | 40분 | Block, LINE/WORD, 좌표 추출 |
| 2 | **문서 분석: 폼 & 표** | `analyze_document` | 50분 | KEY_VALUE_SET, TABLE 파싱 |
| 3 | **쿼리 기반 정보 추출** | `analyze_document` (QUERIES) | 40분 | 자연어 질의, 답변 추출 |
| 4 | **영수증 / 인보이스 분석** | `analyze_expense` | 40분 | ExpenseDocument, LineItems |
| 5 | **신분증 분석** | `analyze_id` | 30분 | IdentityDocumentFields |
| 6 | **비동기 처리 & 실전 파이프라인** | `start_document_analysis` | 40분 | S3, JobId, 폴링 패턴 |

---

## 🏗️ Amazon Textract 핵심 개념: Block

Textract는 문서를 **Block** 단위로 분석합니다. 모든 결과가 Block 객체로 반환됩니다.

```
PAGE
├── LINE
│   ├── WORD
│   └── WORD
├── TABLE
│   ├── CELL (row=1, col=1)
│   └── CELL (row=1, col=2)
├── KEY_VALUE_SET (KEY)
│   └── KEY_VALUE_SET (VALUE)
└── QUERY
    └── QUERY_RESULT
```

### Block 주요 필드
| 필드 | 설명 |
|---|---|
| `BlockType` | PAGE, LINE, WORD, TABLE, CELL, KEY_VALUE_SET, QUERY, QUERY_RESULT |
| `Text` | 추출된 텍스트 내용 |
| `Confidence` | 신뢰도 (0~100) |
| `Geometry.BoundingBox` | 위치 정보 (Left, Top, Width, Height — 0~1 비율) |
| `Relationships` | 관련 Block의 Id 목록 (CHILD, VALUE) |
| `Id` | Block 고유 식별자 (UUID) |

---

## 🔧 API 선택 가이드

```
텍스트만 필요?  →  detect_document_text
표/폼 구조?     →  analyze_document (TABLES, FORMS)
특정 항목 추출? →  analyze_document (QUERIES)
영수증/인보이스? → analyze_expense
신분증(여권 등)? → analyze_id
대용량/PDF 다중 페이지? → start_document_* (비동기 + S3)
```


## 🔧 환경 설정 확인

아래 셀을 실행하여 필요한 라이브러리와 AWS 자격증명을 확인하세요.


In [1]:
# ✅ [제공 코드] 라이브러리 임포트 및 버전 확인
import boto3
import json
import os
import time
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd

print("✅ 라이브러리 임포트 완료")
print(f"  boto3 버전: {boto3.__version__}")

# Textract 클라이언트 초기화
textract = boto3.client('textract', region_name='ap-northeast-2')
s3 = boto3.client('s3', region_name='ap-northeast-2')

# 자격증명 확인
try:
    sts = boto3.client('sts')
    identity = sts.get_caller_identity()
    print(f"  AWS 계정 ID: {identity['Account']}")
    print(f"  ARN: {identity['Arn']}")
    print("\n✅ AWS 자격증명 정상")
except Exception as e:
    print(f"\n❌ 자격증명 오류: {e}")
    print("   → AWS 자격증명을 확인하세요 (IAM Role 또는 Access Key)")


✅ 라이브러리 임포트 완료
  boto3 버전: 1.43.0


  AWS 계정 ID: 054422645032
  ARN: arn:aws:sts::054422645032:assumed-role/AmazonSageMaker-ExecutionRole-20260629T114176/SageMaker

✅ AWS 자격증명 정상


In [2]:
# ✅ [제공 코드] 공통 유틸리티 함수 (모든 실습에서 사용)

def load_document_bytes(path):
    """로컬 파일을 바이트로 읽기"""
    with open(path, 'rb') as f:
        return f.read()

def show_image(image_path, title='문서 원본'):
    """이미지 출력"""
    img = Image.open(image_path)
    plt.figure(figsize=(10, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.show()
    return img

def draw_bounding_boxes(image_path, blocks, block_types=None, color_map=None, title='결과'):
    """BoundingBox를 이미지에 그리는 공통 함수"""
    img = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    w, h = img.size
    
    default_colors = {'LINE': 'blue', 'WORD': 'cyan', 'TABLE': 'red',
                      'CELL': 'orange', 'KEY_VALUE_SET': 'green', 'QUERY_RESULT': 'purple'}
    if color_map:
        default_colors.update(color_map)
    
    for block in blocks:
        btype = block.get('BlockType', '')
        if block_types and btype not in block_types:
            continue
        if 'Geometry' not in block:
            continue
        box = block['Geometry']['BoundingBox']
        left   = box['Left'] * w
        top    = box['Top'] * h
        right  = left + box['Width'] * w
        bottom = top  + box['Height'] * h
        color = default_colors.get(btype, 'yellow')
        draw.rectangle([left, top, right, bottom], outline=color, width=2)
    
    plt.figure(figsize=(12, 14))
    plt.imshow(img)
    plt.axis('off')
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.show()

print("✅ 공통 유틸리티 함수 로드 완료")
print("  사용 가능한 함수: load_document_bytes(), show_image(), draw_bounding_boxes()")


✅ 공통 유틸리티 함수 로드 완료
  사용 가능한 함수: load_document_bytes(), show_image(), draw_bounding_boxes()


## 📁 실습 이미지 준비

각 실습에 필요한 이미지를 `./images/` 폴더에 준비하세요.

| 실습 | 파일명 | 추천 문서 유형 |
|---|---|---|
| Lab 1 | `lab01_text.jpg` | 텍스트가 많은 문서, 신문, 책 페이지 |
| Lab 2 | `lab02_form.jpg` | 표나 양식이 있는 문서 (신청서, 설문지) |
| Lab 3 | `lab03_query.jpg` | 계약서, 이력서, 특정 정보가 있는 문서 |
| Lab 4 | `lab04_receipt.jpg` | 영수증, 인보이스, 청구서 |
| Lab 5 | `lab05_id.jpg` | 여권, 운전면허증 (※ 개인정보 주의) |
| Lab 6 | `lab06_async.pdf` | 다중 페이지 PDF |

> **💡 팁**: 인터넷에서 샘플 문서를 다운로드하거나, 직접 문서를 스캔해서 사용해보세요.
